### Upload corpus.jsonl here before running



This code was confusing at first because of Huggingface's definition of "encode".
"Encode" in huggingface is the job of the tokenizer which splits a long sentence into tokens and then into numbers representing those tokens. This is just a lookup from a fixed vocabulary. There is no meaning attached to these tokens. when two tokens have close integer values, that does not imply they are close in meaning.


But here we are getting an "Encoder model" from MedCPT, whose task is to turn the above taken IDs into meanings.
The "Encoder" here is the "Encoder" half of the transformer.
It takes the integer IDs, looks up a learned vector for each, then passes them through layers of attention so every token’s vector gets updated based on the whole context.

For ease this brain-fry I renamed the tokenizer output to "tokenized" instead of the conventional "enc"


In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load Model and Tokenizer
tok = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [2]:
import pandas as pd

corpus = pd.read_json('/content/corpus.jsonl', lines=True)
corpus = corpus[['title','abstract']]
docs = corpus.to_dict(orient='records')
print(f"Loaded {len(docs)} documents")

Loaded 57984 documents


In [3]:
from tqdm.auto import tqdm

batch_size = 32
all_vecs = []

# Process all documents in batches with a progress bar
for i in tqdm(range(0, len(docs), batch_size), desc="Encoding documents"):
    batch_docs = docs[i:i + batch_size]
    # Create title + abstract pairs
    batch_texts = [[d["title"], d["abstract"]] for d in batch_docs]

    with torch.no_grad():
        tokenized = tok(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        # Extract [CLS] representation (first token output) for the batch
        vecs = model(**tokenized).last_hidden_state[:, 0, :]
        all_vecs.append(vecs.cpu())

# Concatenate all batch vectors into a single tensor
all_vecs = torch.cat(all_vecs, dim=0)
print(f"Encoded {len(docs)} documents into shape {all_vecs.shape}")

Encoding documents:   0%|          | 0/1812 [00:00<?, ?it/s]

Encoded 57984 documents into shape torch.Size([57984, 768])


In [4]:
# Save the encoded document embeddings tensor to a file
torch.save(all_vecs, 'corpus_embeddings.pt')
print("Saved embeddings tensor to 'corpus_embeddings.pt'")

Saved embeddings tensor to 'corpus_embeddings.pt'


### Now download corpus_embeddings.pt